# start

In [1394]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [1395]:
# to be cleaned: games_data.csv, steamspy_raw.json, game_ids.csv, and game-level review summary
ROOT = Path.cwd().parent # works only if this file is kept in a subdirectory of root
RAW_DATA_PATH = ROOT / "data" / "raw"
RAW_GAME_IDS_FILE = RAW_DATA_PATH / "game_ids.csv"
RAW_GAMES_DATA_FILE = RAW_DATA_PATH / "games_data.csv"
RAW_STEAMSPY_FILE = RAW_DATA_PATH / "steamspy_raw.json"
RAW_STEAM_REVIEWS_FILE = RAW_DATA_PATH / "steam_reviews_summary.csv"

# games_id.csv cleaning

In [1396]:
df = pd.read_csv(RAW_GAME_IDS_FILE)
df[df.isna().any(axis=1)]

Empty DataFrame
Columns: [appid, name, source_page]
Index: []

no NaN values, so this file should be good

# games_data.csv cleaning

## removing missing values

There are a couple rows with NaN values.
- rows with a NaN `price` and NaN `release_date` means that it is delisted.
- rows with a NaN `price` but with a release date means that it's not available for purchase, either from being taken off or from being part of a bigger bundle.

In [1397]:
df = pd.read_csv(RAW_GAMES_DATA_FILE)
condition = (df.isna().any(axis=1))
df[condition]

       appid  type                             name  \
0     453480  game                  Shadowverse CCG   
1     466240  game                           Deceit   
7    1818750  game                      MultiVersus   
13     63380  game                  Sniper Elite V2   
37   1063730  game              New World: Aeternum   
45     17480  game  Command & Conquer™ Red Alert™ 3   
50   1811260  game               EA SPORTS™ FIFA 23   
63    681660  game                     Bless Online   
89   1894790  game     Stones Keeper: King Aurelius   
97   2558700  game             The Backrooms Deluxe   
114   432260  game                         Tiltagon   
137  1703080  game                F-15 Strike Eagle   
142   699760  game             Triple Otakus Puzzle   
147  1595770  game                         SUFFER 2   
154  1686540  game                   my fluffy life   

                                                genres  \
0                               Strategy; Free To Play   
1  

Let's just remove them

In [1398]:
print("rows before:", len(df))
df = df[~condition]
print("rows after:", len(df))

rows before: 200
rows after: 185


## removing duplicates

In [1399]:
df[df.duplicated()]

Empty DataFrame
Columns: [appid, type, name, genres, categories, initial_price, final_price, is_free, release_date, description]
Index: []

there doesn't seem to be any duplicates

## fixing invalid data types

In [1400]:
df.dtypes

appid              int64
type                 str
name                 str
genres               str
categories           str
initial_price    float64
final_price      float64
is_free             bool
release_date         str
description          str
dtype: object

`release_date` should be a datetime object

In [1401]:
df['release_date'] = pd.to_datetime(df['release_date'])
df.dtypes

appid                     int64
type                        str
name                        str
genres                      str
categories                  str
initial_price           float64
final_price             float64
is_free                    bool
release_date     datetime64[us]
description                 str
dtype: object

## fixing inconsistent formatting
the `genres` and `categories` columns' datatypes were originally strings, and the list of genres was just a single string with semicolons in it. This will turn them into arrays

In [1402]:
df['genres_list'] = df['genres'].str.split(r'\s*;\s*')
df['categories_list'] = df['categories'].str.split(r'\s*;\s*')
if not df['genres_list'].isna().all() and not df['categories_list'].isna().all():
    df['genres'] = df.pop('genres_list')
    df['categories'] = df.pop('categories_list')


df

       appid  type                                             name  \
2     252490  game                                             Rust   
3     424370  game                          Wolcen: Lords of Mayhem   
4     356190  game                    Middle-earth™: Shadow of War™   
5    1716740  game                                        Starfield   
6    1238840  game                                   Battlefield™ 1   
..       ...   ...                                              ...   
195   774531  game         Snail Trek - Chapter 4: The Final Fondue   
196  3205190  game                                  Alice's Lullaby   
197   682450  game                              Garrison: Archangel   
198   680680  game                                     All Haze Eve   
199  2658720  game  Vampire: The Masquerade - Reckoning of New York   

                                                genres  \
2    [Action, Adventure, Indie, Massively Multiplay...   
3                      [Action,

## TODO standardizing genres
I plan to create a new column called `Primary Genre` that has the main genre a game is.
- There are a couple of genres in the genre list that are not needed, such as Indie and Casual, however there are games where their only listed genre is one of either genre. I will just remove them.

In [1403]:
genres_condition = (df['genres'].str.len() == 1) & (
    (df['genres'].apply(lambda x: "Casual" in x)) | (df['genres'].apply(lambda x: "Indie" in x))
)

df[genres_condition]

       appid  type                         name    genres  \
22    470220  game                          UNO  [Casual]   
68   2198680  game              Dead Town Tales   [Indie]   
79    717950  game                Rainbow Snake  [Casual]   
86    836450  game          Heaven Will Be Mine   [Indie]   
129  1359900  game                 Gone Golfing   [Indie]   
141   701880  game                 Rocka Feller   [Indie]   
146  1382310  game                     Splinter   [Indie]   
171  1395830  game                      Hoplegs   [Indie]   
172   698030  game        Comit the Astrodian 3   [Indie]   
183   968290  game  Deported 2: Build That Wall   [Indie]   

                                            categories  initial_price  \
22   [Single-player, Multi-player, PvP, Online PvP,...           9.99   
68   [Single-player, Steam Achievements, Steam Clou...          11.99   
79   [Single-player, Steam Achievements, Family Sha...           2.99   
86   [Single-player, Steam Achieveme

In [1404]:
df = df[~genres_condition]

## standardizing categories
Categories have either a "single-player" or "multi-player" or both in their array. I'll make new columns that is either true or false for these categories

In [1405]:
df["singleplayer"] = df["categories"].apply(lambda x: "Single-player" in x)
df["multiplayer"] = df["categories"].apply(lambda x: "Multi-player" in x)


There's a program named Soundpad that is erroneously listed as a game, and isn't listed as either singleplayer nor multiplayer, so I'll remove it

In [1406]:
df[(df['singleplayer'] == False) & (df['multiplayer'] == False)]

     appid  type      name                         genres        categories  \
30  629520  game  Soundpad  [Audio Production, Utilities]  [Family Sharing]   

    initial_price  final_price  is_free release_date  \
30           4.99         4.99    False   2017-09-30   

                                          description  singleplayer  \
30  Play sounds in voice chats in high digital qua...         False   

    multiplayer  
30        False  

In [1407]:
df = df[~((df['singleplayer'] == False) & (df['multiplayer'] == False))]

In [1408]:
print("rows: ", df.shape[0])

rows:  174


# steam_reviews_summary.csv cleaning
- Since we're going to combine data from this and `games_data.csv` together, I'll make a separate df. 

In [1409]:
reviews_df = pd.read_csv(RAW_STEAM_REVIEWS_FILE)
reviews_df

       appid  review_score review_score_desc  total_positive  total_negative  \
0     453480             6   Mostly Positive           11116            3951   
1     466240             6   Mostly Positive           72772           19288   
2     252490             8     Very Positive         1221414          188010   
3     424370             5             Mixed           38021           30036   
4     356190             8     Very Positive          104749           13012   
..       ...           ...               ...             ...             ...   
195   774531             7          Positive              44               0   
196  3205190             7          Positive              34               6   
197   682450             8     Very Positive             208              41   
198   680680             6   Mostly Positive               8               3   
199  2658720             5             Mixed             252             125   

     total_reviews  positive_percentage

In [1410]:
reviews_df.dtypes

appid                    int64
review_score             int64
review_score_desc          str
total_positive           int64
total_negative           int64
total_reviews            int64
positive_percentage    float64
collected_at               str
dtype: object

the `collected_at` column only shows when each row was collected by us, so we can remove it

In [1411]:
reviews_df = reviews_df.drop(columns=['collected_at'])
reviews_df

       appid  review_score review_score_desc  total_positive  total_negative  \
0     453480             6   Mostly Positive           11116            3951   
1     466240             6   Mostly Positive           72772           19288   
2     252490             8     Very Positive         1221414          188010   
3     424370             5             Mixed           38021           30036   
4     356190             8     Very Positive          104749           13012   
..       ...           ...               ...             ...             ...   
195   774531             7          Positive              44               0   
196  3205190             7          Positive              34               6   
197   682450             8     Very Positive             208              41   
198   680680             6   Mostly Positive               8               3   
199  2658720             5             Mixed             252             125   

     total_reviews  positive_percentage

# combining steam_reviews and games_data

In [1412]:
merged = pd.merge(df, reviews_df, on='appid')
merged

       appid  type                                             name  \
0     252490  game                                             Rust   
1     424370  game                          Wolcen: Lords of Mayhem   
2     356190  game                    Middle-earth™: Shadow of War™   
3    1716740  game                                        Starfield   
4    1238840  game                                   Battlefield™ 1   
..       ...   ...                                              ...   
169   774531  game         Snail Trek - Chapter 4: The Final Fondue   
170  3205190  game                                  Alice's Lullaby   
171   682450  game                              Garrison: Archangel   
172   680680  game                                     All Haze Eve   
173  2658720  game  Vampire: The Masquerade - Reckoning of New York   

                                                genres  \
0    [Action, Adventure, Indie, Massively Multiplay...   
1                      [Action,

# steamspy_raw.json cleaning

In [1413]:
raw = pd.read_json(RAW_STEAMSPY_FILE)
steamspy_df = pd.DataFrame(raw["records"].tolist())
steamspy_df

       appid                                             name  \
0     453480                                  Shadowverse CCG   
1     466240                                           Deceit   
2     252490                                             Rust   
3     424370                          Wolcen: Lords of Mayhem   
4     356190                      Middle-earth: Shadow of War   
..       ...                                              ...   
195   774531         Snail Trek - Chapter 4: The Final Fondue   
196  3205190                                  Alice's Lullaby   
197   682450                              Garrison: Archangel   
198   680680                                     All Haze Eve   
199  2658720  Vampire: The Masquerade - Reckoning of New York   

                developer                          publisher score_rank  \
0           Cygames, Inc.  Cygames, Inc., Cygames Korea Inc.              
1            World Makers                       World Makers         

There are a couple columns that are useless:
- average and median columns and userscore column are always 0
- `score_rank` is empty
- Languages
- genre lists are already in another dataframe
- I believe that `ccu` only gets how many players are playing when the data was collected

In [1414]:
steamspy_df = steamspy_df.drop(columns=['average_forever', 'average_2weeks', 'median_forever', 'median_2weeks', 'languages', 'genre', 'ccu', 'userscore', 'score_rank'])

## fix prices formatting
- for some reason, prices do not have decimal points and are stored as strings

In [1415]:
steamspy_df['price'] = steamspy_df['price'].astype('int64')
steamspy_df['price'] /= 100

steamspy_df['initialprice'] = steamspy_df['initialprice'].astype('int64')
steamspy_df['initialprice'] /= 100

steamspy_df['discount'] = steamspy_df['discount'].astype('int64')

steamspy_df

       appid                                             name  \
0     453480                                  Shadowverse CCG   
1     466240                                           Deceit   
2     252490                                             Rust   
3     424370                          Wolcen: Lords of Mayhem   
4     356190                      Middle-earth: Shadow of War   
..       ...                                              ...   
195   774531         Snail Trek - Chapter 4: The Final Fondue   
196  3205190                                  Alice's Lullaby   
197   682450                              Garrison: Archangel   
198   680680                                     All Haze Eve   
199  2658720  Vampire: The Masquerade - Reckoning of New York   

                developer                          publisher  positive  \
0           Cygames, Inc.  Cygames, Inc., Cygames Korea Inc.     10839   
1            World Makers                       World Makers     73606 

## owners estimates
- owners of steam games are only measured in a range between two numbers (which is in a string). we need to fix that and find a good way to represent the "amount" of owners in one number

In [1416]:
bounds = (
    steamspy_df["owners"]
    .str.replace(",", "", regex=False)
    .str.split(r"\s*\.\.\s*", expand=True)
    .astype("int64")
)

steamspy_df["owners_low"] = bounds[0]
steamspy_df["owners_high"] = bounds[1]

steamspy_df["owners_mean"] = (steamspy_df["owners_low"] + steamspy_df["owners_high"]) / 2
steamspy_df["owners_mean"] = steamspy_df["owners_mean"].round().astype("int64")

target_idx = steamspy_df.columns.get_loc('owners')
steamspy_df.insert(target_idx, "owners_mean", steamspy_df.pop('owners_mean'))
# steamspy_df.insert(target_idx, "owners_high", steamspy_df.pop('owners_high'))
# steamspy_df.insert(target_idx, "owners_low", steamspy_df.pop('owners_low'))
steamspy_df = steamspy_df.drop(columns=['owners', 'owners_low', 'owners_high'])

steamspy_df

       appid                                             name  \
0     453480                                  Shadowverse CCG   
1     466240                                           Deceit   
2     252490                                             Rust   
3     424370                          Wolcen: Lords of Mayhem   
4     356190                      Middle-earth: Shadow of War   
..       ...                                              ...   
195   774531         Snail Trek - Chapter 4: The Final Fondue   
196  3205190                                  Alice's Lullaby   
197   682450                              Garrison: Archangel   
198   680680                                     All Haze Eve   
199  2658720  Vampire: The Masquerade - Reckoning of New York   

                developer                          publisher  positive  \
0           Cygames, Inc.  Cygames, Inc., Cygames Korea Inc.     10839   
1            World Makers                       World Makers     73606 

## tags
- genres are picked by the publisher, while tags are voted by users. 
    - we can remove some unnecessary tags and get the highest rated tag for each game

In [1417]:
# #old code
# bad_tags = {
#     "Free to Play", 
#     "PvP", 
#     "Singleplayer", 
#     "Multiplayer", 
#     "Open World", 
#     "Action", 
#     "Casual", 
#     "Cyberpunk", 
#     "Stealth", 
#     "Space",
#     "Co-op",
# }

# steamspy_df["tags"] = steamspy_df["tags"].apply(
#     lambda t: {k: v for k, v in t.items() if k not in bad_tags} if isinstance(t, dict) else t
# )

# steamspy_df["top_tag"] = steamspy_df["tags"].apply(
#     lambda t: max(t, key=t.get) if isinstance(t, dict) and t else None
# )
# steamspy_df

In [ ]:
# GENRES = {
#     # Action
#     "Action", "Shooter", "FPS", "Third-Person Shooter", "Platformer", "Precision Platformer",
#     "Metroidvania", "Hack and Slash", "Beat 'em up", "Fighting", "Bullet Hell",
#     "Shoot 'Em Up", "Roguelike", "Roguelite", "Action Roguelike", "Souls-like",
#     "Stealth", "Survival", "Survival Horror", "Battle Royale", "Arena Shooter",
#     "Twin Stick Shooter", "Rhythm", "Spectacle fighter",
#     # RPG
#     "RPG", "Action RPG", "JRPG", "CRPG", "Tactical RPG", "Strategy RPG", "Party-Based RPG",
#     "Dungeon Crawler", "MMORPG", "Roguelike Deckbuilder",
#     # Strategy
#     "Strategy", "Turn-Based Strategy", "Real Time Strategy", "RTS", "Grand Strategy",
#     "4X", "Tower Defense", "Auto Battler", "Card Game", "Deckbuilding", "Board Game",
#     "Wargame", "MOBA", "City Builder", "Colony Sim", "Base Building", "Turn-Based Tactics",
#     # Simulation / management
#     "Simulation", "Farming Sim", "Life Sim", "Management", "Automation", "Building",
#     "Driving", "Racing", "Flight", "Space Sim", "Sports", "Immersive Sim",
#     "Sandbox", "Open World Survival Craft", "Crafting",
#     # Story / adventure
#     "Adventure", "Visual Novel", "Interactive Fiction", "Point & Click", "Walking Simulator",
#     "Dating Sim", "Choose Your Own Adventure", "Mystery", "Detective",
#     # Puzzle / casual
#     "Puzzle", "Puzzle Platformer", "Match 3", "Hidden Object", "Word Game", "Trivia",
#     "Idler", "Clicker", "Incremental",
#     # Other
#     "Horror", "Psychological Horror", "Party Game", "Arcade", "Pinball", "Typing",
# }

# def top_genre(t):
#     if not isinstance(t, dict):
#         return None
#     genre_tags = {k: v for k, v in t.items() if k in GENRES}
#     return max(genre_tags, key=genre_tags.get) if genre_tags else None

# steamspy_df["tag_main"] = steamspy_df["tags"].apply(top_genre)
# steamspy_df

In [ ]:
GENRE_GROUP = {
    # Shooter
    "Shooter": "Shooter", "FPS": "Shooter", "Third-Person Shooter": "Shooter",
    "Arena Shooter": "Shooter", "Battle Royale": "Shooter", "Twin Stick Shooter": "Shooter",
    # Action
    "Action": "Action", "Platformer": "Action", "Precision Platformer": "Action",
    "Metroidvania": "Action", "Hack and Slash": "Action", "Beat 'em up": "Action",
    "Fighting": "Action", "Spectacle fighter": "Action", "Souls-like": "Action",
    "Stealth": "Action", "Arcade": "Action", "Shoot 'Em Up": "Action", "Pinball": "Action",
    "Rhythm": "Action",
    # Roguelike
    "Roguelike": "Roguelike", "Roguelite": "Roguelike", "Action Roguelike": "Roguelike",
    "Roguelike Deckbuilder": "Roguelike", "Bullet Hell": "Roguelike",
    # RPG
    "RPG": "RPG", "Action RPG": "RPG", "JRPG": "RPG", "CRPG": "RPG", "Tactical RPG": "RPG",
    "Strategy RPG": "RPG", "Party-Based RPG": "RPG", "Dungeon Crawler": "RPG", "MMORPG": "RPG",
    # Strategy
    "Strategy": "Strategy", "Turn-Based Strategy": "Strategy", "Real Time Strategy": "Strategy",
    "RTS": "Strategy", "Grand Strategy": "Strategy", "4X": "Strategy", "Tower Defense": "Strategy",
    "Auto Battler": "Strategy", "Card Game": "Strategy", "Deckbuilding": "Strategy",
    "Board Game": "Strategy", "Wargame": "Strategy", "MOBA": "Strategy",
    "Turn-Based Tactics": "Strategy",
    # Survival & Sandbox
    "Survival": "Survival & Sandbox", "Open World Survival Craft": "Survival & Sandbox",
    "Sandbox": "Survival & Sandbox", "Crafting": "Survival & Sandbox",
    "Base Building": "Survival & Sandbox", "Colony Sim": "Survival & Sandbox",
    "Building": "Survival & Sandbox",
    # Simulation
    "Simulation": "Simulation", "Farming Sim": "Simulation", "Life Sim": "Simulation",
    "Management": "Simulation", "Automation": "Simulation", "City Builder": "Simulation",
    "Driving": "Simulation", "Racing": "Simulation", "Flight": "Simulation",
    "Space Sim": "Simulation", "Sports": "Simulation", "Immersive Sim": "Simulation",
    # Adventure / Story
    "Adventure": "Adventure", "Visual Novel": "Adventure", "Interactive Fiction": "Adventure",
    "Point & Click": "Adventure", "Walking Simulator": "Adventure", "Dating Sim": "Adventure",
    "Choose Your Own Adventure": "Adventure", "Mystery": "Adventure", "Detective": "Adventure",
    # Horror
    "Horror": "Horror", "Survival Horror": "Horror", "Psychological Horror": "Horror",
    # Puzzle / Casual
    "Puzzle": "Puzzle", "Puzzle Platformer": "Puzzle", "Match 3": "Puzzle",
    "Hidden Object": "Puzzle", "Word Game": "Puzzle", "Trivia": "Puzzle",
    "Idler": "Puzzle", "Clicker": "Puzzle", "Incremental": "Puzzle",
    "Party Game": "Puzzle", "Typing": "Puzzle",
}

steamspy_df["tag_raw"] = steamspy_df["tags"].apply(top_genre)
steamspy_df["tag_main"] = steamspy_df["tag_raw"].map(GENRE_GROUP).fillna("Other")


Some values are NaN if they don't fit any of the genres, so we can remove them

In [1419]:
steamspy_df[steamspy_df.isnull().any(axis=1)]

       appid                  name      developer      publisher  positive  \
30    629520              Soundpad       Leppsoft       Leppsoft     84918   
79    717950         Rainbow Snake     RandomSpin     RandomSpin       133   
141   701880          Rocka Feller      BBQ Games      BBQ Games         1   
142   699760  Triple Otakus Puzzle  Gamesforgames  Gamesforgames        25   
146  1382310              Splinter   Hobo Studios   Hobo Studios         3   

     negative  owners_mean  price  initialprice  discount  \
30       3466      3500000   4.99          4.99         0   
79         48        75000   0.49          0.99        51   
141         0        10000   1.99          1.99         0   
142        53        10000   0.49          0.99        51   
146         0        10000   0.99          0.99         0   

                                                  tags tag_raw tag_main  
30   {'Utilities': 173, 'Audio Production': 163, 'M...     NaN    Other  
79              

In [1420]:
steamspy_df = steamspy_df[~steamspy_df.isnull().any(axis=1)]
print(f"rows now: {steamspy_df.shape[0]}")

rows now: 195


# Combining All the Files
There are three dataframes to combine:
- df, which is the games_data.csv 
- reviews_df
- steamspy_df

Some columns are shared between them so we can pick what to exclude

In [1421]:
df.columns.to_list()

['appid',
 'type',
 'name',
 'genres',
 'categories',
 'initial_price',
 'final_price',
 'is_free',
 'release_date',
 'description',
 'singleplayer',
 'multiplayer']

In [1422]:
reviews_df.columns.to_list()

['appid',
 'review_score',
 'review_score_desc',
 'total_positive',
 'total_negative',
 'total_reviews',
 'positive_percentage']

In [1423]:
steamspy_df.columns.to_list()

['appid',
 'name',
 'developer',
 'publisher',
 'positive',
 'negative',
 'owners_mean',
 'price',
 'initialprice',
 'discount',
 'tags',
 'tag_raw',
 'tag_main']

In [1424]:
merged = (
    df
    .merge(reviews_df, on="appid")
    .merge(steamspy_df[["appid", "developer", "publisher", "owners_mean", "tag_main"]], on="appid")
)

merged

       appid  type                                             name  \
0     252490  game                                             Rust   
1     424370  game                          Wolcen: Lords of Mayhem   
2     356190  game                    Middle-earth™: Shadow of War™   
3    1716740  game                                        Starfield   
4    1238840  game                                   Battlefield™ 1   
..       ...   ...                                              ...   
169   774531  game         Snail Trek - Chapter 4: The Final Fondue   
170  3205190  game                                  Alice's Lullaby   
171   682450  game                              Garrison: Archangel   
172   680680  game                                     All Haze Eve   
173  2658720  game  Vampire: The Masquerade - Reckoning of New York   

                                                genres  \
0    [Action, Adventure, Indie, Massively Multiplay...   
1                      [Action,

# Statistics

## Genres by count

In [1425]:
merged['tag_main'].value_counts()

tag_main
Action                35
Adventure             27
Strategy              26
RPG                   17
Simulation            16
Horror                14
Survival & Sandbox    12
Shooter               11
Puzzle                10
Roguelike              6
Name: count, dtype: int64